In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))
from src.data import load_data
from src.train import train_model
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
import numpy as np
import pandas as pd


# Load Data

In [ ]:
path = "../data/raw/train.csv"
df_raw = load_data(path)
# df = df.drop(columns=['column_name'])
y = df_raw["SalePrice"]
X = df_raw.drop(columns=['Id', 'SalePrice'])

# Preprocessing

In [78]:
categorical_features = [
    "MSSubClass",
    "MSZoning",
    "Street",
    "Alley",
    "LotShape",
    "LandContour",
    "Utilities",
    "LotConfig",
    "LandSlope",
    "Neighborhood",
    "Condition1",
    "Condition2",
    "BldgType",
    "HouseStyle",
    "RoofStyle",
    "RoofMatl",
    "Exterior1st",
    "Exterior2nd",
    "MasVnrType",
    "ExterQual",
    "ExterCond",
    "Foundation",
    "BsmtQual",
    "BsmtCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2",
    "Heating",
    "HeatingQC",
    "CentralAir",
    "Electrical",
    "KitchenQual",
    "Functional",
    "FireplaceQu",
    "GarageType",
    "GarageFinish",
    "GarageQual",
    "GarageCond",
    "PavedDrive",
    "PoolQC",
    "Fence",
    "MiscFeature",
    "MoSold",
    "SaleType",
    "SaleCondition",
]

In [79]:
numeric_features = ['LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal', 'YrSold']

In [80]:
numeric_transformer = Pipeline(
    steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
    ]
)
categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)



# Validation Split

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print("X_train shape: ", X_train.shape)
print("X_val shape: ", X_val.shape)
print("y_train shape: ", y_train.shape)
print("y_val shape: ", y_val.shape)

X_train shape:  (1168, 79)
X_val shape:  (292, 79)
y_train shape:  (1168,)
y_val shape:  (292,)


# Model Comparison

In [82]:
lr_model = LinearRegression()
ridge_model = Ridge(alpha=0.1)
lasso_model = Lasso(alpha=0.1)
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
gb_model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)

In [83]:

models = {
    "LinearRegression": {"model_object": lr_model, "RMSE": None},
    "Ridge": {"model_object": ridge_model, "RMSE": None},
    "Lasso": {"model_object": lasso_model, "RMSE": None},
    "RandomForest": {"model_object": rf_model, "RMSE": None},
    "GradientBoosting": {"model_object": gb_model, "RMSE": None}
}
y_train_log = np.log(y_train)
for key in models:
    model_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model',models[key]["model_object"])
    ])
    model_pipeline.fit(X_train, y_train_log)
    y_predict_log =model_pipeline.predict(X_val)
    y_predict = np.exp(y_predict_log)
    models[key]["RMSE"] = root_mean_squared_error(y_val, y_predict)


In [84]:
df = pd.DataFrame(models).T
print(df[["RMSE"]])

                          RMSE
LinearRegression  23220.991423
Ridge             23215.858625
Lasso             52361.847066
RandomForest       30175.07892
GradientBoosting   29686.69758


# Cross-Validation